In [1]:
from aiohttp.web_middlewares import middleware
# 配置环境变量
from dotenv import load_dotenv
from langchain.agents.middleware import SummarizationMiddleware

load_dotenv()

import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == 'your_groq_api_key_here':
    raise ValueError("请将Groq API Key填写到.env文件中")
grop_model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
if not api_key or api_key == 'your_dashscope_api_key_here':
    raise ValueError("请将DashScope API Key填写到.env文件中")
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

# 1.定义简单的工具

In [2]:
@tool
def get_order_status(order_id: str) -> str:
    """
    查询订单状态

    参数：
        order_id: 订单id

    返回：
        字符串类型的订单状态
    """
    orders = {
        "12345": "已发货，预计明天送达",
        "67890": "配送中，今天下午送达"
    }
    return orders.get(order_id, "订单不存在")

# 2.客服机器人

In [3]:
def example():

    db_path = "customer_service.sqlite"
    with SqliteSaver.from_conn_string(db_path) as checkpointer:
        agent = create_agent(
            model=model,
            tools=[get_order_status],
            checkpointer=checkpointer,
            system_prompt="""
                你是一个客服助手
                特点：
                - 记住客户之间的咨询
                - 友好、耐心
                - 能利用工具查询订单
            """,
            middleware=[SummarizationMiddleware(
                model=grop_model,
                trigger=("tokens", 800)
            )]
        )
        customer_id = "user_1234"
        config = {"configurable": {"thread_id": customer_id}}
        print("第一次查询")
        conversation_1 = [
            "你好，我想查询订单",
            "订单号是12345"
        ]
        for msg in conversation_1:
            print(f"\n客户：{msg}")
            response = agent.invoke(
                {"messages": [{"role": "user", "content": msg}]},
                config=config
            )
            print(f"客服：{response['messages'][-1].content}")

        print("\n第二次查询")
        print("\n客户：我的订单到哪了？")
        response = agent.invoke(
            {"messages": [{"role": "user", "content": "我的订单到哪了？"}]},
            config=config
        )
        print(f"客服：{response['messages'][-1].content}")

In [4]:
def main():
    print("\n" + "="*70)
    print("checkpointing")
    print("="*70)

    try:
        example()
    except KeyboardInterrupt:
        print("\n程序中断")
    except Exception as e:
        print(f"错误： {e}")
        import traceback
        traceback.print_exc()


In [5]:
if __name__ == "__main__":
    main()


checkpointing
第一次查询

客户：你好，我想查询订单
客服：您好！很高兴为您服务。请提供一下您的订单ID，我来帮您查询订单状态。

客户：订单号是12345
客服：您的订单12345已经发货，预计明天就可以送达。如果有其他问题或需要进一步的帮助，请随时告诉我！

第二次查询

客户：我的订单到哪了？
客服：您的订单12345目前状态为已发货，预计明天送达。如果您需要更详细的物流信息，可能需要查看物流公司提供的跟踪链接或访问其官方网站进行查询。如果我这里有权限获取更多详情的话，我会帮您进一步确认的。还有其他我可以协助您的吗？
